# DATASCAPE — tahap 1 pseudo-label otomatis (CPU)

Menghasilkan usulan label dari probabilitas OOF yang sudah ada. Tidak perlu GPU.
Label asli tidak ditimpa, tidak ada manual labeling, dan test set tidak dibaca.

**Hasil ini belum membuktikan peningkatan macro-F1. Jangan memakai proposal OOF global untuk student CV pada fold yang sama.** Baca `reports/pseudo_label_stage1.md` untuk eksperimen nested A/B.

AI Declaration: OpenAI Codex membantu implementasi dan pengujian audit. Semua usulan berasal dari probabilitas model yang dilatih pada train resmi; tidak ada keputusan label oleh manusia.


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts/pseudo_label_audit.py').is_file():
    raise FileNotFoundError('Buka notebook dari root repo DATASCAPE, atau set ROOT ke lokasi repo.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.pseudo_label_audit import run
MANIFEST = ROOT / 'artifacts/preprocessing_colab/training_manifest.csv'
PROBABILITIES = ROOT / '.validation/imported_result/oof_probabilities.npy'
SOURCE_SUMMARY = ROOT / '.validation/imported_result/run_summary.json'
OUTPUT = ROOT / 'artifacts/pseudo_label_stage1'


In [ ]:
summary = run(MANIFEST, PROBABILITIES, SOURCE_SUMMARY, OUTPUT)
print(json.dumps({k: summary[k] for k in ['rows', 'disagreements', 'threshold_pass', 'proposals', 'student_trained']}, indent=2))
display(pd.read_csv(OUTPUT / 'proposed_transitions.csv'))


In [ ]:
candidates = pd.read_csv(OUTPUT / 'pseudo_label_candidates.csv')
display(candidates[['filepath', 'label_original', 'label_proposed', 'confidence', 'original_probability', 'margin', 'decision']])
assert (candidates.label == candidates.label_original).all()
assert not candidates.student_cv_allowed.any()


## Eksperimen berikutnya

`nested_evaluation_plan.csv` menyimpan pembagian data untuk eksperimen baru. Teacher inner hanya boleh melihat outer-train, dengan epoch tetap atau tuning tambahan di dalam data teacher. Outer-validation selalu memakai label asli dan tidak dipakai untuk relabel/teacher selection. Student A dan B memakai model/seed/transform/budget sama; hanya label train yang berbeda. Probabilitas OOF global lama tidak boleh dipakai untuk melatih pasangan student CV tersebut.

MobileNetV3-Large dapat dipakai untuk kedua student agar hemat komputasi. Tidak ada training yang dijalankan oleh notebook audit ini.


In [ ]:
plan = pd.read_csv(OUTPUT / 'nested_evaluation_plan.csv')
display(pd.crosstab([plan.outer_role, plan.inner_fold], plan.label))
print('Protokol lengkap:', ROOT / 'reports/pseudo_label_stage1.md')
